# Price deep-learning baselines (Darts)
notebook for deep learning methods

reports main results: MAE, RMSE, aQL

In [ ]:
!pip install -r requirements/darts.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

REPO_ROOT=Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT/"requirements").exists():
    REPO_ROOT=REPO_ROOT.parent
ROOT=REPO_ROOT/"price"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0,str(REPO_ROOT))
if str(ROOT/"src") not in sys.path:
    sys.path.insert(0,str(ROOT/"src"))

from sklearn import preprocessing

from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import TFTModel, TSMixerModel, RNNModel, TiDEModel, DLinearModel, NLinearModel
from darts.utils.likelihood_models import QuantileRegression

from preprocessing import load_data, FUTURE_COVARIATES, PAST_COVARIATES
from gefcom import (
    load_gefcom_price,
    evaluate_gefcom_price,
    PRICE_COVARIATES,
    GEFCOM_PRICE_ALL_DATES,
)
from src.metrics import print_metrics

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))

MODEL_CLASSES={
    "tft":TFTModel,
    "tsmixer":TSMixerModel,
    "deepar":RNNModel,
    "tide":TiDEModel,
    "dlinear":DLinearModel,
    "nlinear":NLinearModel,
}

OUTPUT_DIR=ROOT/"outputs"/"predictions"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)


## GEFCom2014-P


In [ ]:
# GEFCom2014-P (cleaned input; no extra cleaning)
from gefcom import load_gefcom_price, price_context_and_days, evaluate_gefcom_price, PRICE_COVARIATES
gef_price=load_gefcom_price(ROOT/"data"/"gefcom_price_data.csv")
gef_context, gef_test, gef_days=price_context_and_days(gef_price)
GEF_OUTPUT=ROOT/"outputs"/"predictions"; GEF_OUTPUT.mkdir(parents=True,exist_ok=True)


In [ ]:
GEF_MODEL="tide"  # tft, tsmixer, deepar, tide, dlinear, nlinear
GEF_LOAD_EXISTING=True
GEF_MODEL_DIR=ROOT/"models"/"darts"/"gefcom"
GEF_MODEL_DIR.mkdir(parents=True,exist_ok=True)
GEF_MODEL_PATH=GEF_MODEL_DIR/f"{GEF_MODEL}.darts"

gef_df=load_gefcom_price(ROOT/"data"/"gefcom_price_data.csv")

gef_series=TimeSeries.from_dataframe(
    gef_df,
    time_col="timestamp",
    value_cols="target",
    freq="h",
).astype(np.float32)

gef_future_covariates=TimeSeries.from_dataframe(
    gef_df,
    time_col="timestamp",
    value_cols=PRICE_COVARIATES,
    freq="h",
).astype(np.float32)

gef_cutoff=pd.Timestamp("2013-06-15 23:00:00")
gef_train,gef_test=gef_series.split_after(gef_cutoff)
gef_train,gef_val=gef_train.split_after(0.8)

gef_target_scale=Scaler(preprocessing.StandardScaler())
gef_train_t=gef_target_scale.fit_transform(gef_train)
gef_val_t=gef_target_scale.transform(gef_val)
gef_series_t=gef_target_scale.transform(gef_series)

gef_future_train,gef_future_test=gef_future_covariates.split_after(gef_cutoff)
gef_future_train,gef_future_val=gef_future_train.split_after(0.8)

gef_future_scale=Scaler(preprocessing.StandardScaler())
gef_future_train_t=gef_future_scale.fit_transform(gef_future_train)
gef_future_val_t=gef_future_scale.transform(gef_future_val)
gef_future_cov_t=gef_future_scale.transform(gef_future_covariates)

print("GEF target:", gef_series_t.start_time(), "->", gef_series_t.end_time())
print("GEF future covariates:", gef_future_cov_t.start_time(), "->", gef_future_cov_t.end_time())
assert gef_future_cov_t.end_time() >= gef_series_t.end_time()

if GEF_LOAD_EXISTING:
    if not GEF_MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {GEF_MODEL_PATH}. "
            "Put the GEFCom checkpoint there or set GEF_LOAD_EXISTING=False."
        )
    gef_model=MODEL_CLASSES[GEF_MODEL].load(
        str(GEF_MODEL_PATH),
        weights_only=False,
    )
else:
    likelihood=QuantileRegression(BASE_QUANTILES)
    common=dict(
        input_chunk_length=168,
        output_chunk_length=24,
        likelihood=likelihood,
        n_epochs=100,
        random_state=42,
        force_reset=True,
    )
    if GEF_MODEL=="tft":
        gef_model=TFTModel(**common,hidden_size=64,lstm_layers=1,num_attention_heads=4)
    elif GEF_MODEL=="tsmixer":
        gef_model=TSMixerModel(**common,hidden_size=64)
    elif GEF_MODEL=="deepar":
        gef_model=RNNModel(
            model="LSTM",
            input_chunk_length=168,
            output_chunk_length=24,
            training_length=1000,
            hidden_dim=64,
            likelihood=likelihood,
            n_epochs=100,
            random_state=42,
            force_reset=True,
        )
    elif GEF_MODEL=="tide":
        gef_model=TiDEModel(**common,hidden_size=64)
    elif GEF_MODEL=="dlinear":
        gef_model=DLinearModel(**common)
    elif GEF_MODEL=="nlinear":
        gef_model=NLinearModel(**common,normalize=False)
    else:
        raise ValueError(GEF_MODEL)

    gef_model.fit(
        gef_train_t,
        val_series=gef_val_t,
        future_covariates=gef_future_train_t,
        val_future_covariates=gef_future_val_t,
        verbose=True,
    )
    gef_model.save(str(GEF_MODEL_PATH))


In [ ]:
gef_backtests=gef_model.historical_forecasts(
    series=gef_series_t,
    future_covariates=gef_future_cov_t,
    start=gef_test.start_time(),
    forecast_horizon=24,
    stride=24,
    num_samples=100,
    last_points_only=False,
    retrain=False,
    verbose=True,
)

gef_forecast_t=concatenate(gef_backtests)
gef_forecast=gef_target_scale.inverse_transform(gef_forecast_t)
gef_gt=gef_series.slice_intersect(gef_forecast)
gef_forecast=gef_forecast.slice_intersect(gef_gt)

gef_result=pd.DataFrame({
    "item_id":1,
    "timestamp":gef_gt.time_index,
    "target":gef_gt.values().flatten(),
})
for q in BASE_QUANTILES:
    gef_result[str(q)]=gef_forecast.quantile(q).values().flatten()
gef_result["prediction"]=gef_result["0.5"]

gef_result["task_date"]=pd.to_datetime(gef_result["timestamp"]).dt.floor("D")
gef_result=gef_result[
    gef_result["task_date"].isin(GEFCOM_PRICE_ALL_DATES)
].drop(columns="task_date").reset_index(drop=True)

expected=24*len(GEFCOM_PRICE_ALL_DATES)
if len(gef_result)!=expected:
    raise ValueError(
        f"Expected {expected} GEFCom forecast rows (15 x 24), got {len(gef_result)}."
    )

gef_result.to_csv(
    OUTPUT_DIR/f"{GEF_MODEL}_gefcom_price.csv",
    index=False,
)

gef_summary=evaluate_gefcom_price(gef_result,BASE_QUANTILES)
display(gef_summary[["task_date","raw_aql","normalized_aql"]])
gef_summary.to_csv(
    OUTPUT_DIR/f"{GEF_MODEL}_gefcom_price_summary.csv",
    index=False,
)


## GridStatus2025


In [ ]:
MODEL="tide"  # tft, tsmixer, deepar, tide, dlinear, nlinear
NON_GEF_LOAD_EXISTING=True
MODEL_DIR=ROOT/"models"/"darts"/"gridstatus"
MODEL_DIR.mkdir(parents=True,exist_ok=True)
MODEL_PATH=MODEL_DIR/f"{MODEL}.darts"

data=load_data(ROOT/"data")

series=TimeSeries.from_dataframe(
    data.full,
    time_col="timestamp",
    value_cols="target",
    freq="h",
).astype(np.float32)

future_covariates=TimeSeries.from_dataframe(
    data.full,
    time_col="timestamp",
    value_cols=FUTURE_COVARIATES,
    freq="h",
).astype(np.float32)

past_covariates=TimeSeries.from_dataframe(
    data.full,
    time_col="timestamp",
    value_cols=PAST_COVARIATES,
    freq="h",
).astype(np.float32)

testing_cutoff=pd.Timestamp("2024-12-31 23:00:00")

train,test=series.split_after(testing_cutoff)
train,val=train.split_after(0.8)

target_scale=Scaler(preprocessing.StandardScaler())
train_t=target_scale.fit_transform(train)
val_t=target_scale.transform(val)
test_t=target_scale.transform(test)
series_t=target_scale.transform(series)

past_train,past_test=past_covariates.split_after(testing_cutoff)
past_train,past_val=past_train.split_after(0.8)
past_scale=Scaler(preprocessing.StandardScaler())
past_train_t=past_scale.fit_transform(past_train)
past_val_t=past_scale.transform(past_val)
past_test_t=past_scale.transform(past_test)
past_cov_t=past_scale.transform(past_covariates)

future_train,future_test=future_covariates.split_after(testing_cutoff)
future_train,future_val=future_train.split_after(0.8)
future_scale=Scaler(preprocessing.StandardScaler())
future_train_t=future_scale.fit_transform(future_train)
future_val_t=future_scale.transform(future_val)
future_test_t=future_scale.transform(future_test)
future_cov_t=future_scale.transform(future_covariates)

print("Target:", series_t.start_time(), "->", series_t.end_time())
print("Past covariates:", past_cov_t.start_time(), "->", past_cov_t.end_time())
print("Future covariates:", future_cov_t.start_time(), "->", future_cov_t.end_time())

if past_cov_t.end_time() < testing_cutoff:
    raise ValueError(
        "GridStatus past covariates do not reach the test boundary. "
        f"They end at {past_cov_t.end_time()}, but must reach at least {testing_cutoff}. "
        "Check Exogenous 4 in lmp_NP15_data.csv."
    )
if future_cov_t.end_time() < series_t.end_time():
    raise ValueError(
        "GridStatus future covariates do not cover the full target series. "
        f"They end at {future_cov_t.end_time()}, target ends at {series_t.end_time()}."
    )

if NON_GEF_LOAD_EXISTING:
    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {MODEL_PATH}. "
            "Put the GridStatus checkpoint there or set NON_GEF_LOAD_EXISTING=False."
        )
    model=MODEL_CLASSES[MODEL].load(
        str(MODEL_PATH),
        weights_only=False,
    )
else:
    likelihood=QuantileRegression(OUTPUT_QUANTILES)
    common=dict(
        input_chunk_length=168,
        output_chunk_length=24,
        likelihood=likelihood,
        n_epochs=100,
        random_state=42,
        force_reset=True,
    )
    if MODEL=="tft":
        model=TFTModel(**common,hidden_size=64,lstm_layers=1,num_attention_heads=4)
    elif MODEL=="tsmixer":
        model=TSMixerModel(**common,hidden_size=64)
    elif MODEL=="deepar":
        model=RNNModel(
            model="LSTM",
            input_chunk_length=168,
            output_chunk_length=24,
            training_length=1000,
            hidden_dim=64,
            likelihood=likelihood,
            n_epochs=100,
            random_state=42,
            force_reset=True,
        )
    elif MODEL=="tide":
        model=TiDEModel(**common,hidden_size=64)
    elif MODEL=="dlinear":
        model=DLinearModel(**common)
    elif MODEL=="nlinear":
        model=NLinearModel(**common,normalize=False)
    else:
        raise ValueError(MODEL)

    fit_kwargs=dict(
        series=train_t,
        val_series=val_t,
        future_covariates=future_train_t,
        val_future_covariates=future_val_t,
        verbose=True,
    )
    if MODEL!="deepar":
        fit_kwargs.update(
            past_covariates=past_train_t,
            val_past_covariates=past_val_t,
        )
    model.fit(**fit_kwargs)
    model.save(str(MODEL_PATH))


In [ ]:
start_time=test.start_time()
# if MODEL=="deepar":

#     start_time=start_time + pd.Timedelta(hours=24)

hf_kwargs=dict(
    series=series_t,
    future_covariates=future_cov_t,
    start=start_time,
    forecast_horizon=24,
    stride=24,
    num_samples=100,
    last_points_only=False,
    retrain=False,
    verbose=True,
)
if MODEL!="deepar":
    hf_kwargs["past_covariates"]=past_cov_t

backtest_series=model.historical_forecasts(**hf_kwargs)

forecast_t=concatenate(backtest_series)
forecast=target_scale.inverse_transform(forecast_t)
gt=series.slice_intersect(forecast)
forecast=forecast.slice_intersect(gt)

result=pd.DataFrame({
    "item_id":1,
    "timestamp":gt.time_index,
    "target":gt.values().flatten(),
})
for q in OUTPUT_QUANTILES:
    result[str(q)]=forecast.quantile(q).values().flatten()
result["prediction"]=result["0.5"]

result.to_csv(
    OUTPUT_DIR/f"{MODEL}.csv",
    index=False,
)
print_metrics(result,BASE_QUANTILES,label=MODEL)
